DATA CHECKING

In [ ]:
import re
import pandas as pd

# File location
file_path = r"D:\muba\MID.xlsx"

print("--- 1. LOADING DATASET ---")
try:
    df = pd.read_excel(file_path)
    print(f"Dataset loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns.\n")
except Exception as e:
    print(f"Error loading file: {e}")
    # Install openpyxl if missing: pip install openpyxl
    raise SystemExit()

# --- 2. COLUMNS & DATA TYPES ---
print("--- 2. COLUMN LIST & TYPES ---")
print(df.dtypes)
print("\n")

# --- 3. MISSING VALUE AUDIT ---
print("--- 3. MISSING VALUE AUDIT ---")
missing_df = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
print(missing_df.sort_values(by='Missing Count', ascending=False))
print("\n")

# --- 4. DUPLICATES AUDIT ---
print("--- 4. DUPLICATES AUDIT ---")
print(f"Total Exact Duplicate Rows: {df.duplicated().sum()}")
if 'Name' in df.columns:
    print(f"Duplicate Drug Names ('Name' column): {df.duplicated(subset=['Name']).sum()}")
print("\n")

# --- 5. TEXT ARTIFACT & NOISE AUDIT ---
print("--- 5. NOISE & HTML TAG AUDIT ---")
target_cols = ['Name', 'Contains', 'Product Uses', 'Safety Advice', 'Therapeutic Class']

for col in target_cols:
    if col in df.columns:
        col_str = df[col].astype(str)
        html_tags = col_str.str.contains(r'<[^>]+>', regex=True, na=False).sum()
        escapes = col_str.str.contains(r'\\[ntr]', regex=True, na=False).sum()
        string_nulls = col_str.str.lower().isin(['null', 'none', 'nan', '']).sum()
        
        print(f"Column [{col}]:")
        print(f"  • Rows with HTML tags (<p>, <ul>, etc.): {html_tags}")
        print(f"  • Rows with escape artifacts (\\n, \\t, etc.): {escapes}")
        print(f"  • Rows with text 'null' or empty strings: {string_nulls}")

# --- 6. SAMPLE INSPECTION ---
print("\n--- 6. SAMPLE HEAD ROWS ---")
print(df[['Name', 'Contains', 'Therapeutic Class']].head(5))

Loading dataset...
Cleaning text columns and removing HTML tags...

SUCCESS: File saved to 'd:\Downloads\Microsoft VS Code\cleaned_medicines.csv' with 147852 rows!


DATA CLEANING

In [ ]:
import re
import os
import pandas as pd

# 1. Load dataset
file_path = r"D:\muba\MID.xlsx"
print("Loading dataset...")
df = pd.read_excel(file_path)


# Cleaning helper function to strip HTML tags, escape characters, and whitespace
def clean_text(val):
    if pd.isna(val) or str(val).strip().lower() in [
        "null",
        "none",
        "nan",
        "",
        "null\n",
    ]:
        return ""
    text = str(val)
    # Remove HTML tags (e.g., <p dir="ltr">, <ul>)
    text = re.sub(r"<[^>]+>", " ", text)
    # Remove raw escape artifacts (\n, \t, \r, \', \")
    text = re.sub(r"\\[ntr\'\"]", " ", text)
    # Collapse extra whitespace
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 2. Apply cleaning across all text fields
target_cols = [
    "Name",
    "Contains",
    "ProductIntroduction",
    "ProductUses",
    "ProductBenefits",
    "SideEffect",
    "HowToUse",
    "HowWorks",
    "QuickTips",
    "SafetyAdvice",
    "Chemical_Class",
    "Habit_Forming",
    "Therapeutic_Class",
    "Action_Class",
]

print("Cleaning text columns and removing HTML tags...")
for col in target_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# 3. Drop rows missing essential drug names or active ingredients
df = df.dropna(subset=["Name", "Contains"])
df = df[(df["Name"] != "") & (df["Contains"] != "")]

# 4. Drop exact duplicate medicine entries
df = df.drop_duplicates(subset=["Name"])

# 5. Select core columns for MediFind AI pipeline
cleaned_df = df[
    [
        "Name",
        "Contains",
        "Therapeutic_Class",
        "Action_Class",
        "Chemical_Class",
        "ProductUses",
        "SideEffect",
        "SafetyAdvice",
        "Habit_Forming",
    ]
]

# 6. Save clean file to CSV (much faster to load than Excel for the app)
output_csv = "cleaned_medicines.csv"
cleaned_df.to_csv(output_csv, index=False)

print(
    f"\nSUCCESS: File saved to '{os.path.abspath(output_csv)}' with {len(cleaned_df)} rows!"
)

DATA CLEANING DEAL WITH MISSING VALUE

In [3]:
import os
import pandas as pd

# 1. Define file paths in Documents directory
user_documents = os.path.join(os.path.expanduser("~"), "Documents")
input_csv = os.path.join(user_documents, "cleaned_medicines.csv")
output_csv = os.path.join(user_documents, "cleaned_medicines_final.csv")

# 2. Load dataset (fallback to local directory if file isn't in Documents)
if os.path.exists(input_csv):
    df = pd.read_csv(input_csv)
else:
    df = pd.read_csv("cleaned_medicines.csv")

# 3. Drop rows missing essential identifiers
df = df.dropna(subset=["Name", "Contains"])

# 4. Impute empty cells with default fallback text
df["Therapeutic_Class"] = (
    df["Therapeutic_Class"].fillna("Unclassified").replace("", "Unclassified")
)
df["Action_Class"] = (
    df["Action_Class"]
    .fillna("General Specialty")
    .replace("", "General Specialty")
)
df["Chemical_Class"] = (
    df["Chemical_Class"].fillna("Unclassified").replace("", "Unclassified")
)

df["ProductUses"] = (
    df["ProductUses"]
    .fillna("Consult a doctor for specific indications.")
    .replace("", "Consult a doctor for specific indications.")
)
df["SideEffect"] = (
    df["SideEffect"]
    .fillna(
        "No major side effects recorded. Consult a physician if symptoms persist."
    )
    .replace(
        "",
        "No major side effects recorded. Consult a physician if symptoms persist.",
    )
)
df["SafetyAdvice"] = (
    df["SafetyAdvice"]
    .fillna("Use under professional medical supervision.")
    .replace("", "Use under professional medical supervision.")
)
df["Habit_Forming"] = df["Habit_Forming"].fillna("No").replace("", "No")

# 5. Catch-all for any remaining empty cells
df = df.fillna("Information not available").replace(
    "", "Information not available"
)

# 6. Save final dataset
df.to_csv(output_csv, index=False)

print(
    f"SUCCESS: Saved final dataset to '{output_csv}' with {len(df)} clean rows!"
)
print(f"Remaining null cells count: {df.isnull().sum().sum()}")

SUCCESS: Saved final dataset to 'C:\Users\OWNER\Documents\cleaned_medicines_final.csv' with 147852 clean rows!
Remaining null cells count: 0


GENERIC DRUG LOOKUP ENGINE

In [5]:
import pandas as pd

df = pd.read_csv("cleaned_medicines_final.csv")


def find_generic_substitutes(search_term, top_n=5):
    # Search for medicine name
    match = df[
        df["Name"].str.contains(search_term, case=False, na=False)
    ]

    if match.empty:
        return f"No medicine found matching '{search_term}'", []

    # Get primary active ingredient
    target_medicine = match.iloc[0]["Name"]
    active_ingredient = match.iloc[0]["Contains"]

    # Find substitutes sharing the exact active ingredient
    substitutes = df[
        (df["Contains"] == active_ingredient)
        & (df["Name"] != target_medicine)
    ]["Name"].unique()

    return {
        "searched_medicine": target_medicine,
        "active_ingredient": active_ingredient,
        "generic_substitutes": list(substitutes[:top_n]),
    }


# Test query
print(find_generic_substitutes("Augmentin"))

{'searched_medicine': 'Augmentin 625 Duo Tablet', 'active_ingredient': 'Amoxycillin (500mg)+ Clavulanic Acid (125mg)', 'generic_substitutes': ['Amoxyclav 625 Tablet', 'Almox-CV 625 Tablet', 'Advent 625 Tablet', 'Acuclav 625 Tablet', 'Augpen 625 BID Tablet']}


BUSINESS LOGIC ENGINE

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("cleaned_medicines_final.csv")

# Global Brand to Active Ingredient Alias Mapping
BRAND_ALIASES = {
    "panadol": "paracetamol",
    "tylenol": "paracetamol",
    "advil": "ibuprofen",
    "nurofen": "ibuprofen",
    "lipitor": "atorvastatin",
    "glucophage": "metformin",
}

# Define store availability categories
OTC_CHAINS = [
    "Watsons",
    "Guardian",
    "Caring Pharmacy",
    "Local Community Pharmacies",
]
RX_CHAINS = [
    "Licensed Community Pharmacies (Prescription Required)",
    "Hospital Dispensaries",
]
SPECIALTY_CHAINS = ["Major Hospital Pharmacies & Medical Centers Only"]

OTC_INGREDIENTS = [
    "paracetamol",
    "ibuprofen",
    "cetirizine",
    "fexofenadine",
    "loratadine",
]


def get_store_availability(active_ingredient, brand_name):
    ingredient_lower = str(active_ingredient or "").lower()
    brand_lower = str(brand_name or "").lower()

    # 1. Hospital / Specialty drugs (Injections, Biologics)
    if any(
        kw in brand_lower for kw in ["injection", "infusion", "iv", "avastin"]
    ):
        return SPECIALTY_CHAINS, "Specialty / Hospital Only"

    # 2. Over-the-Counter (OTC)
    if any(otc in ingredient_lower for otc in OTC_INGREDIENTS):
        return OTC_CHAINS, "Over-The-Counter (OTC)"

    # 3. Standard Prescription (Rx)
    return RX_CHAINS, "Prescription Required (Rx)"


def smart_medicine_search(query, top_n=5):
    raw_query = str(query or "").strip()
    clean_query = raw_query.lower()

    if clean_query in BRAND_ALIASES:
        clean_query = BRAND_ALIASES[clean_query]

    # Search by Brand Name
    brand_matches = df[
        df["Name"].str.contains(clean_query, case=False, na=False)
    ]

    # Fallback to Active Ingredient column
    if brand_matches.empty:
        brand_matches = df[
            df["Contains"].str.contains(clean_query, case=False, na=False)
        ]

    if brand_matches.empty:
        return {
            "query": raw_query,
            "status": "Not Found",
            "matched_brand": "N/A",
            "active_ingredient": "N/A",
            "availability_type": "N/A",
            "available_at": "N/A",
            "generics_count": 0,
            "substitutes": [],
        }

    # Sort: Prioritize non-injections and single-ingredient drugs
    brand_matches = brand_matches.copy()
    brand_matches["is_injection"] = brand_matches["Name"].str.contains(
        "Injection|Infusion|IV", case=False, na=False
    )
    brand_matches["ingredient_count"] = brand_matches[
        "Contains"
    ].str.count(r"\+")

    sorted_matches = brand_matches.sort_values(
        by=["is_injection", "ingredient_count"]
    )

    top_match = sorted_matches.iloc[0]
    matched_brand = top_match["Name"]
    active_ingredient = top_match["Contains"]

    # Pull generic substitutes
    substitutes = df[
        (df["Contains"] == active_ingredient)
        & (df["Name"] != matched_brand)
    ]["Name"].unique()

    # Determine store locations
    stores, avail_type = get_store_availability(
        active_ingredient, matched_brand
    )

    return {
        "query": raw_query,
        "status": "Success",
        "matched_brand": matched_brand,
        "active_ingredient": active_ingredient,
        "availability_type": avail_type,
        "available_at": ", ".join(stores),
        "generics_count": len(substitutes),
        "substitutes": list(substitutes[:top_n]),
    }


# --- BENCHMARK TEST SUITE ---
test_medicines = [
    "Panadol",
    "Augmentin",
    "Allegra",
    "Azithral",
    "Amoxyclav",
    "Atorvastatin",
    "Metformin",
    "Omeprazole",
    "Ibuprofen",
    "Avastin",
]

# Run benchmark query list
results = [smart_medicine_search(q) for q in test_medicines]
benchmark_df = pd.DataFrame(results)

# --- ROW-BY-ROW PRINTING ---
print("\n" + "=" * 65)
print("             MEDICINE & STORE AVAILABILITY BENCHMARK             ")
print("=" * 65)

for index, row in benchmark_df.iterrows():
    print(f"[{index + 1}] Query: {row['query']}")
    print(f"    Status           : {row['status']}")
    print(f"    Matched Brand    : {row['matched_brand']}")
    print(f"    Active Ingredient: {row['active_ingredient']}")
    print(f"    Category         : {row['availability_type']}")
    print(f"    Available At     : {row['available_at']}")
    print(f"    Generics Count   : {row['generics_count']}")
    if row["substitutes"]:
        print(f"    Top Substitutes  : {', '.join(row['substitutes'][:3])}")
    print("-" * 65)
    


             MEDICINE & STORE AVAILABILITY BENCHMARK             
[1] Query: Panadol
    Status           : Success
    Matched Brand    : Genericart Paracetamol 650mg Tablet
    Active Ingredient: Paracetamol (650mg)
    Category         : Over-The-Counter (OTC)
    Available At     : Watsons, Guardian, Caring Pharmacy, Local Community Pharmacies
    Generics Count   : 335
    Top Substitutes  : Admol 650 Tablet DT, Arden 650mg Tablet, Algina 650 Tablet
-----------------------------------------------------------------
[2] Query: Augmentin
    Status           : Success
    Matched Brand    : Augmentin 625 Duo Tablet
    Active Ingredient: Amoxycillin (500mg)+ Clavulanic Acid (125mg)
    Category         : Prescription Required (Rx)
    Available At     : Licensed Community Pharmacies (Prescription Required), Hospital Dispensaries
    Generics Count   : 722
    Top Substitutes  : Amoxyclav 625 Tablet, Almox-CV 625 Tablet, Advent 625 Tablet
---------------------------------------------